Second-Hand Car Recommendation: Model Training

Cell 1 - Imports and configuration

In [16]:
import json
import re
from pathlib import Path
import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import KFold, cross_val_score, train_test_split
from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeRegressor

SEED = 42
TEST_SIZE = 0.70
INPUT_PATH = Path('data/car_dataset.csv')
ARTIFACT_DIR = Path('artifacts')
REPORT_DIR = Path('reports')
ARTIFACT_DIR.mkdir(exist_ok=True)
REPORT_DIR.mkdir(exist_ok=True)
sns.set_theme(style='whitegrid')

Cell 2 - Load the Kaggle CSV

In [17]:
if not INPUT_PATH.exists():
    raise FileNotFoundError(f'Put the dataset at {INPUT_PATH}')
raw = pd.read_csv(INPUT_PATH)
print('Shape:', raw.shape)
display(raw.head())

FileNotFoundError: Put the dataset at data/car_dataset.csv

Cell 3 - Normalize columns and clean data

In [ ]:
def canonical(value):
    return re.sub(r'[^a-z0-9]+', '_', str(value).strip().lower()).strip('_')

def numeric_series(series):
    return pd.to_numeric(series.astype(str).str.replace(r'[₹$€£,\\s]', '', regex=True).str.extract(r'([-+]?\\d*\\.?\\d+)')[0], errors='coerce')

data = raw.copy()
data.columns = [canonical(c) for c in data.columns]

target_aliases = ['selling_price', 'selling_price', 'price', 'sellingprice', 'amount', 'cost']
target = next((c for c in data.columns if c in target_aliases), None)


if target is None:
    raise ValueError(f'No price column found. Columns: {list(data.columns)}')

data[target] = numeric_series(data[target])
data = data.dropna(subset=[target])
data = data[data[target] > 0]

for column in data.columns:

    if column != target and data[column].dtype == 'object':
        parsed = numeric_series(data[column])

        if parsed.notna().mean() >= 0.8:
            data[column] = parsed
            
drop = [c for c in data.columns if c in {'id', 'car_id', 'unnamed_0', 'index', 'url', 'image', 'description'}]
data = data.drop(columns=drop, errors='ignore').drop_duplicates().reset_index(drop=True)
features = [c for c in data.columns if c != target and data[c].notna().any()]
if len(data) < 10 or not features:
    raise ValueError('Need at least 10 rows and one usable feature.')
print('Target:', target, '| Rows:', len(data), '| Features:', features)
display(data.describe(include='all').T)

ValueError: Need at least 10 rows and one usable feature.

Cell 4 - Exploratory dataset analysis

In [ ]:
numeric_columns = data.select_dtypes(include=np.number).columns
display(data[numeric_columns].corr()[[target]].sort_values(target, ascending=False))
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.histplot(data[target], kde=True, ax=axes[0], color='#4472c4')
axes[0].set_title('Price distribution')
sns.boxplot(x=data[target], ax=axes[1], color='#70ad47')
axes[1].set_title('Price outlier overview')
plt.tight_layout()
plt.savefig(REPORT_DIR / 'dataset_analysis.png', dpi=150)
plt.show()

Cell 5 - Split data into 30% training and 70% testing

In [ ]:
X = data[features]
y = data[target]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=TEST_SIZE, random_state=SEED)
print(f'Train rows: {len(X_train)} ({len(X_train)/len(X):.0%})')
print(f'Test rows: {len(X_test)} ({len(X_test)/len(X):.0%})')

Cell 6 - Leakage-safe preprocessing and four model definitions

In [ ]:
numeric_features = X.select_dtypes(include=np.number).columns.tolist()
categorical_features = [c for c in X.columns if c not in numeric_features]
preprocessor = ColumnTransformer([
    ('numeric', Pipeline([('imputer', SimpleImputer(strategy='median')), ('scale', StandardScaler())]), numeric_features),
    ('categorical', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('onehot', OneHotEncoder(handle_unknown='ignore'))]), categorical_features),
])
models = {
    'random_forest': RandomForestRegressor(n_estimators=300, max_depth=14, min_samples_leaf=2, max_features=0.8, random_state=SEED, n_jobs=-1),
    'decision_tree': DecisionTreeRegressor(max_depth=10, min_samples_leaf=3, min_samples_split=6, random_state=SEED),
    'linear_regression': Ridge(alpha=10.0),
    'knn': KNeighborsRegressor(n_neighbors=min(7, max(1, len(X_train)-1)), weights='distance'),
}
pipelines = {name: Pipeline([('preprocess', preprocessor), ('model', model)]) for name, model in models.items()}
print('Models:', list(pipelines))

Cell 7 - Train models and calculate test metrics

In [ ]:
predictions, metrics, cv_scores = {}, {}, {}
cv = KFold(n_splits=min(5, len(X_train)), shuffle=True, random_state=SEED)
for name, pipeline in pipelines.items():
    pipeline.fit(X_train, y_train)
    prediction = pipeline.predict(X_test)
    predictions[name] = prediction
    metrics[name] = {'r2': r2_score(y_test, prediction), 'mae': mean_absolute_error(y_test, prediction), 'rmse': np.sqrt(mean_squared_error(y_test, prediction))}
    cv_scores[name] = float(np.mean(cross_val_score(pipeline, X_train, y_train, cv=cv, scoring='r2')))
pd.DataFrame(metrics).T.round(4)

Cell 8 - Build the all-model ensemble

In [ ]:
positive = {name: max(score, 0.0) for name, score in cv_scores.items()}
total = sum(positive.values())
weights = {name: (positive[name] / total if total else 1 / len(pipelines)) for name in pipelines}
ensemble_prediction = sum(weights[name] * predictions[name] for name in pipelines)
metrics['ensemble'] = {'r2': r2_score(y_test, ensemble_prediction), 'mae': mean_absolute_error(y_test, ensemble_prediction), 'rmse': np.sqrt(mean_squared_error(y_test, ensemble_prediction))}
print('Cross-validation R2:', cv_scores)
print('Ensemble weights:', weights)
display(pd.DataFrame(metrics).T.round(4))

Cell 9 - Individual model plots and comparison plot

In [ ]:
for name, prediction in predictions.items():
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].scatter(y_test, prediction, alpha=0.55)
    low, high = min(y_test.min(), prediction.min()), max(y_test.max(), prediction.max())
    axes[0].plot([low, high], [low, high], 'r--')
    axes[0].set(xlabel='Actual price', ylabel='Predicted price', title=f'{name}: actual vs predicted')
    axes[1].hist(y_test.to_numpy() - prediction, bins=25, color='#4472c4')
    axes[1].set(xlabel='Residual', title='Error distribution')
    fig.tight_layout(); fig.savefig(REPORT_DIR / f'{name}.png', dpi=150); plt.show()
comparison = pd.DataFrame(metrics).T
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for axis, metric, title in zip(axes, ['r2', 'mae', 'rmse'], ['R2 (higher)', 'MAE (lower)', 'RMSE (lower)']):
    comparison[metric].plot.bar(ax=axis, color='#4472c4', title=title)
    axis.set_xlabel('')
fig.tight_layout(); fig.savefig(REPORT_DIR / 'model_comparison.png', dpi=150); plt.show()

Cell 10 - Save artifacts for the FastAPI backend

In [ ]:
joblib.dump({'models': pipelines, 'weights': weights, 'features': features, 'target': target}, ARTIFACT_DIR / 'model_bundle.joblib')
data.to_csv(ARTIFACT_DIR / 'cleaned_data.csv', index=False)
summary = {'target': target, 'rows': len(data), 'features': features, 'train_rows': len(X_train), 'test_rows': len(X_test), 'test_size': TEST_SIZE, 'metrics': {k: {m: round(float(v), 6) for m, v in vals.items()} for k, vals in metrics.items()}, 'cross_validation_r2': cv_scores, 'ensemble_weights': weights}
(ARTIFACT_DIR / 'metrics.json').write_text(json.dumps(summary, indent=2))
print('Saved model_bundle.joblib, cleaned_data.csv, metrics.json, and all report plots.')